- Create spark session

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("silver_transformations").getOrCreate()

In [0]:
df = spark.table("teleco_churn_datalakehouse.bronze.teleco_customer_churn")
type(df)

- Dropping duplicate customerId's

In [0]:
# Remove any duplicate customerId's
from pyspark.sql.functions import col
df = df.dropDuplicates(["CustomerID"])

- Confirm duplicate customer ID's

In [0]:
#  Confirm for duplicates in the CustomerID

# Create a temporary view of the the dataframe we are working on.
df.createOrReplaceTempView("churn_table")

spark.sql("""
          SELECT CustomerID, COUNT(*) AS repeated
          FROM churn_table
          GROUP BY CustomerID
          HAVING COUNT(*) > 1
""").show()

- Trim all columns of datatype "StringType"

In [0]:
# Trimming all string/varchar columns
from pyspark.sql.functions import trim, col

string_cols = [field.name for field in df.schema.fields if str(field.dataType) == 'StringType']
for c in string_cols:
    df = df.withColumn(c, trim(col(c)))

- Filling and confirming presence of null values in State , City and Country

In [0]:
# Filling null values
df = df.fillna({"Country":"n/a", "State":"n/a", "City":"n/a" })

# check out for null locations
is_there_null = df.filter(col("Country").isNull()).show()
is_there_null = df.filter(col("State").isNull()).show()
is_there_null = df.filter(col("City").isNull()).show()


- Checking for and filling null zip code values

In [0]:
# Check for zip code nil status
spark.sql("""
          SELECT CustomerID, Zip_Code
          FROM teleco_churn_datalakehouse.bronze.teleco_customer_churn
          WHERE Zip_Code IS NULL OR Zip_Code < 0 ; 
""").show()


# Fill in null values in the Zip-Code column
df = df.fillna({"Zip_Code":0})

- Checking for and filling Gender , Senior_citizen , patener , dependants

In [0]:
# Check for null gender , senior citizen status , patner and dependents
spark.sql("""
          SELECT CustomerID
          FROM teleco_churn_datalakehouse.bronze.teleco_customer_churn
          WHERE Gender IS NULL OR Senior_Citizen IS NULL OR Partner IS NULL OR Dependents IS NULL ; 
""").show()

# filling null values for gender , senior_citizen status , patner and dependants
df = df.fillna({"Gender":"n/a", "Senior_Citizen":"n/a", "Partner":"n/a", "Dependents":"n/a"})

- Checking for and filling Tenure_months null values

In [0]:
# Check for null tenure months
spark.sql("""
          SELECT CustomerID
          FROM teleco_churn_datalakehouse.bronze.teleco_customer_churn
          WHERE Tenure_Months IS NULL OR Tenure_Months < 0 ; 
""").show()

# Findings , no null values

# Fill in null values in the Tenure_Months column
df = df.fillna({"Tenure_Months":0})

- Loading the data as a delta table to the silver layer.

In [0]:
df.write.mode("overwrite").saveAsTable("teleco_churn_datalakehouse.silver.teleco_customer_churn")

- Check the detail for the table

In [0]:
%sql
DESCRIBE DETAIL teleco_churn_datalakehouse.silver.teleco_customer_churn

- Confirm data has been loaded to the silver layer

In [0]:
%sql
SELECT * FROM teleco_churn_datalakehouse.silver.teleco_customer_churn LIMIT 500 ; 